# Lecture 4. Comparing two groups

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aaubs/AQM-2026/blob/main/lecture04/lecture04.ipynb)

Applied Quantitative Methods, Fall 2026. Readings: Newbold, Carlson and Thorne, sections 10.1 to 10.4.

This notebook follows the lecture slides. Every test is worked through in the same steps: the question, what we know
and want to find out (including the two hypotheses), the formula and the decision rule, the calculation by hand (on
the slides), the same calculation in Python, and the answer for a manager. In Python we first follow the formula line
by line and then check the result with a ready-made function. Where Python does something differently from Newbold,
or where there are several approaches with different results, a section "... in Python and in Newbold" says so.

Parts 1 to 4 each end with an exercise. In an exercise you predict a result, run the cell, change one value, and
answer a question about what you see. You do not need to write any code. The last section holds a summary table, the
Python functions with their defaults, the symbols and the Danish terms, so you can use the notebook as a reference
afterwards.

Run the setup cell first. It loads the RetailCorp data from the course repository.

## Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

url = "https://raw.githubusercontent.com/aaubs/AQM-2026/main/data/"
sample = pd.read_csv(url + "orders_sample.csv")          # the study of 200 orders
population = pd.read_csv(url + "orders_population.csv")  # all 50,000 orders, used only in demonstrations
survey = pd.read_csv(url + "survey.csv")                 # 300 customers, before and after the redesign
news = pd.read_csv(url + "newsletter_test.csv")          # newsletter test, 5,000 recipients per version

print(f"Orders: {len(sample)}. Survey: {len(survey)} customers. Newsletter: {len(news):,} recipients.")

## The case and the question

In Lecture 3 we tested one sample against a value RetailCorp claims. Today we compare two groups with each other, or
the same customers before and after a change.

**Is the gap between two groups, or between before and after, larger than chance would produce?**

| RetailCorp question | Data | Part |
|---|---|---|
| Did the website redesign raise satisfaction? | `survey.csv`: 300 customers, before and after | 1 |
| Do Copenhagen orders take longer than Aarhus orders? | `orders_sample.csv`: 80 and 120 orders | 2 |
| Do the two newsletter versions lead to different shares of buyers? | `newsletter_test.csv`: 5,000 per version | 3 |
| Are Copenhagen delivery times more spread out than Aarhus delivery times? | `orders_sample.csv` | 4 |

The tests in Parts 1 to 3 have the same form as in Lecture 3:

test statistic = (estimate − value in H<sub>0</sub>) / standard error.

The estimate is now a difference, so the standard error changes. The F test in Part 4 uses a ratio of two variances
instead.

### Paired or independent?

Ask before you compute: can each value in one group be paired with exactly one value in the other, because it is the
same unit, or a unit matched to it such as a twin? Then the samples are **paired** (dependent). In the survey each row is one customer who rated twice, so the
two scores in a row belong together. In the orders, the Copenhagen orders and the Aarhus orders are different orders,
and no order in one group belongs with an order in the other: the samples are **independent**. The same holds for the
newsletter test, where different recipients got each version.

In [ ]:
survey.head()

In [ ]:
sample[["order_id", "warehouse", "delivery_hours"]].head()

## Part 1. Paired samples

**Question.** RetailCorp redesigned its website. The same 300 customers rated their satisfaction from 1 to 10 before
and after. Did satisfaction go up?

**What we know.** 300 customers, each rated twice. The same customers, so the samples are paired.

**What we want.** Whether the sample gives strong evidence that μ<sub>d</sub>, the average change among all
customers, is above 0. Only "went up" answers the question, so the test is one-sided.

**The change for each customer.** For customer i, d<sub>i</sub> = after<sub>i</sub> − before<sub>i</sub>. This gives
one column of 300 changes. From here on, the paired test is the t test for a mean from Lecture 3, applied to the
changes.

**Hypotheses.** H<sub>0</sub>: μ<sub>d</sub> ≤ 0 &nbsp;&nbsp;&nbsp; H<sub>1</sub>: μ<sub>d</sub> > 0
&nbsp;&nbsp;&nbsp; α = 0.05

**Test statistic.** t = d̄ / (s<sub>d</sub>/√n), where d̄ (d-bar) is the average change, s<sub>d</sub> (s-d) the
standard deviation of the changes, and n the number of pairs: 300 customers with two scores each. If
μ<sub>d</sub> = 0, t follows a t distribution with n − 1 degrees of freedom. This holds when the changes are normal,
or n is large. With 300 pairs, the average change is close to normal even though the scores are whole numbers.

**Decision rule.** Reject H<sub>0</sub> if t > t<sub>n−1, α</sub>, or if the p-value is below α.

In [ ]:
before = survey["satisfaction_before"]
after = survey["satisfaction_after"]
print(f"Average before: {before.mean():.2f}, after: {after.mean():.2f}")
print(f"Standard deviation before: {before.std(ddof=1):.3f}")

d = after - before                      # one change per customer
print(d.value_counts().sort_index())    # how many customers changed by -3, ..., +3 points

In [ ]:
n = len(d)                       # 300 pairs
d_bar = d.mean()                 # d-bar
s_d = d.std(ddof=1)              # s_d
se = s_d / np.sqrt(n)
t_stat = d_bar / se
t_crit = stats.t.ppf(0.95, df=n - 1)
p = stats.t.sf(t_stat, df=n - 1)  # area above t

print(f"d_bar = {d_bar:.3f}, s_d = {s_d:.3f}, standard error = {se:.4f}")
print(f"t = {t_stat:.3f}, crit = {t_crit:.3f}, p = {p:.7f}")

**Decision.** t = 4.892 is far above the critical value 1.650, and p = 0.0000008 is far below 0.05, so we reject
H<sub>0</sub>.

**For the manager.** On average, satisfaction rose by 0.25 points on the 10-point scale. A rise of this size is very
unlikely to be chance (p < 0.001), so the survey gives strong evidence that satisfaction went up among customers in
general. It shows that satisfaction rose after the redesign. Without a comparison group, it cannot show that the
redesign caused the rise.

A check: scipy's function for paired tests gives the same t and p.

In [ ]:
r = stats.ttest_rel(after, before, alternative="greater")
print(r.statistic, r.pvalue)

### Paired tests in Python and in Newbold

* **Order of the columns.** `ttest_rel(after, before)` computes after − before. Swapping the two flips the sign of t.
Say which difference you computed, and write H<sub>1</sub> for that difference.
* **The same test as in Lecture 3.** `ttest_rel(after, before)` gives the same t and p as
`ttest_1samp(after - before, 0)`: a paired test is a one-sample test on the changes.
* **Two-sided by default.** scipy reports a two-sided p-value unless you set `alternative` to `"greater"` or
`"less"`. Set it so that it matches your H<sub>1</sub>.
* **Rows must match.** The pairs are the rows. Sorting or filtering one column on its own breaks the pairs. Keep both
scores of a customer in the same row.
* **Very small p-values** are printed in scientific notation: 8.16e-07 means 0.000000816. Report it as p < 0.001.

In [ ]:
print(stats.ttest_1samp(after - before, popmean=0, alternative="greater"))

### Exercise 1. Subtract the other way round

**Predict.** In the cell, d = before − after. What happens to d̄, to t, and to the direction of H<sub>1</sub> that
matches the question "did satisfaction go up"? Write down your guess.

**Run** the cell. It prints d̄, t and the p-value for each direction of H<sub>1</sub>.

In [ ]:
d_e1 = survey["satisfaction_before"] - survey["satisfaction_after"]   # change to after - before in step 3

n_e1 = len(d_e1)
t_e1 = d_e1.mean() / (d_e1.std(ddof=1) / np.sqrt(n_e1))
print(f"d_bar = {d_e1.mean():.3f}, t = {t_e1:.3f}")
print(f"p if H1 is mu_d > 0: {stats.t.sf(t_e1, df=n_e1 - 1):.7f}")
print(f"p if H1 is mu_d < 0: {stats.t.cdf(t_e1, df=n_e1 - 1):.7f}")

**Change** the first line to `survey["satisfaction_after"] - survey["satisfaction_before"]` and run the cell again.

**Answer.** Does the conclusion depend on which way you subtract? What has to match?

## Part 2. Two independent means

**Question.** Do orders shipped from Copenhagen take longer on average than orders shipped from Aarhus?

**What we know.** 80 Copenhagen orders and 120 Aarhus orders: different orders in each group, so independent samples.

**What we want.** Whether the sample gives strong evidence that μ<sub>Cph</sub> − μ<sub>Aar</sub>, the difference in
average delivery time for all orders, is above 0. One-sided, because the question is whether Copenhagen is slower.

**Why variances add.** Each sample average has its own chance error. The difference of two averages carries both, and
for independent samples they do not cancel. We add the variances and then take the square root:

Var(x̄<sub>1</sub> − x̄<sub>2</sub>) = σ<sub>1</sub>²/n<sub>1</sub> + σ<sub>2</sub>²/n<sub>2</sub>, so with
sample variances SE = √(s<sub>1</sub>²/n<sub>1</sub> + s<sub>2</sub>²/n<sub>2</sub>).

**Hypotheses.** H<sub>0</sub>: μ<sub>Cph</sub> − μ<sub>Aar</sub> ≤ 0 &nbsp;&nbsp;&nbsp; H<sub>1</sub>:
μ<sub>Cph</sub> − μ<sub>Aar</sub> > 0 &nbsp;&nbsp;&nbsp; α = 0.05. Group 1 is Copenhagen, group 2 is Aarhus.

**Test statistic.** t = (x̄<sub>1</sub> − x̄<sub>2</sub>) / √(s<sub>1</sub>²/n<sub>1</sub> + s<sub>2</sub>²/n<sub>2</sub>).
It follows approximately a t distribution with v degrees of freedom, from Newbold's formula 10.11, which Python
computes. This version is called Welch's test. It holds when both populations are normal, or both samples are large.
Reject H<sub>0</sub> if t > t<sub>v, α</sub>.

In [ ]:
wh = sample["warehouse"]
cph = sample.loc[wh == "Copenhagen", "delivery_hours"]
aar = sample.loc[wh == "Aarhus", "delivery_hours"]
print(f"Copenhagen: n = {len(cph)}, x_bar = {cph.mean():.3f}, s = {cph.std(ddof=1):.3f}")
print(f"Aarhus:     n = {len(aar)}, x_bar = {aar.mean():.3f}, s = {aar.std(ddof=1):.3f}")

In [ ]:
n1, n2 = len(cph), len(aar)
v1 = cph.var(ddof=1) / n1          # s1^2 / n1
v2 = aar.var(ddof=1) / n2          # s2^2 / n2
se = np.sqrt(v1 + v2)              # variances add
t_stat = (cph.mean() - aar.mean()) / se
df = (v1 + v2)**2 / (v1**2 / (n1 - 1) + v2**2 / (n2 - 1))   # Newbold 10.11
t_crit = stats.t.ppf(0.95, df=df)
p = stats.t.sf(t_stat, df=df)

print(f"s1^2/n1 = {v1:.4f}, s2^2/n2 = {v2:.4f}")
print(f"se = {se:.4f}, t = {t_stat:.3f}, df = {df:.2f}, crit = {t_crit:.3f}, p = {p:.3f}")

**Decision.** t = 1.975 is above the critical value 1.654, and p = 0.025 is below 0.05, so we reject H<sub>0</sub>.

**For the manager.** In the sample, Copenhagen orders took 2.3 hours longer on average (38.2 against 35.9 hours). At
the 5% level this is evidence that Copenhagen orders take longer on average (p = 0.025). The orders were not assigned
to the warehouses at random, so the test shows that the warehouses differ. It cannot say why.

A check: `ttest_ind` with `equal_var=False` gives the same t, degrees of freedom and p.

In [ ]:
welch = stats.ttest_ind(cph, aar, equal_var=False, alternative="greater")
print(round(welch.statistic, 3), round(welch.df, 2), round(welch.pvalue, 3))

### Three cases in Newbold

Newbold (10.2) has three versions of the test for two independent means. You choose by what you can assume about the
population variances.

| What we assume | Standard error | Distribution |
|---|---|---|
| Variances known | √(σ<sub>1</sub>²/n<sub>1</sub> + σ<sub>2</sub>²/n<sub>2</sub>) | Standard normal (z) |
| Variances unknown, assumed equal (pooled) | √(s<sub>p</sub>²/n<sub>1</sub> + s<sub>p</sub>²/n<sub>2</sub>) | t with n<sub>1</sub> + n<sub>2</sub> − 2 degrees of freedom |
| Variances unknown, not assumed equal (Welch) | √(s<sub>1</sub>²/n<sub>1</sub> + s<sub>2</sub>²/n<sub>2</sub>) | t with v degrees of freedom |

The pooled variance s<sub>p</sub>² = ((n<sub>1</sub> − 1)s<sub>1</sub>² + (n<sub>2</sub> − 1)s<sub>2</sub>²) /
(n<sub>1</sub> + n<sub>2</sub> − 2) is a weighted average of the two sample variances. It is the better estimate if the
two population variances are equal. The cell computes the pooled test and, for comparison, the test with the
population standard deviations, which we know only because the data are simulated.

In [ ]:
sp2 = ((n1 - 1) * cph.var(ddof=1) + (n2 - 1) * aar.var(ddof=1)) / (n1 + n2 - 2)
se_pool = np.sqrt(sp2 / n1 + sp2 / n2)
t_pool = (cph.mean() - aar.mean()) / se_pool
print(f"Pooled: s_p^2 = {sp2:.3f}, se = {se_pool:.4f}, t = {t_pool:.3f}, df = {n1 + n2 - 2}, "
      f"crit = {stats.t.ppf(0.95, df=n1 + n2 - 2):.3f}, p = {stats.t.sf(t_pool, df=n1 + n2 - 2):.3f}")

pop_cph = population.loc[population["warehouse"] == "Copenhagen", "delivery_hours"]
pop_aar = population.loc[population["warehouse"] == "Aarhus", "delivery_hours"]
se_known = np.sqrt(pop_cph.var(ddof=0) / n1 + pop_aar.var(ddof=0) / n2)
z = (cph.mean() - aar.mean()) / se_known
print(f"Variances known: se = {se_known:.3f}, z = {z:.3f}, p = {stats.norm.sf(z):.3f}")

Here the pooled test and Welch's test give almost the same answer, because the two spreads are similar. They can
differ a lot when both the sample sizes and the spreads differ (workshop, Extension 3). We use Welch, because it does not need equal
variances.

### Pooled or Welch in Python

`stats.ttest_ind` assumes equal variances unless you write `equal_var=False`. Without it you get the pooled test. The
default does not appear in your code, so it is easy to miss. In your notebook, say which version you used and why.
Welch is the safe choice. The pooled test is fine if you can argue that the variances are equal. Do not choose by
running an F test first (Part 4).

In [ ]:
pooled = stats.ttest_ind(cph, aar, alternative="greater")
welch = stats.ttest_ind(cph, aar, equal_var=False, alternative="greater")
for name, r in [("pooled", pooled), ("Welch", welch)]:
    print(name, round(r.statistic, 3), round(r.df, 2), round(r.pvalue, 3))

### Demonstration: do the variances really add?

This cell is run in the lecture. It draws 1,000 random samples of 80 Copenhagen and 120 Aarhus orders from the
population and computes the difference of the two averages in each. If the formula is right, the standard deviation
of the 1,000 differences should be close to √(σ<sub>1</sub>²/80 + σ<sub>2</sub>²/120). We can do this only because the
population is simulated.

In [ ]:
reps = 1000
rng = np.random.default_rng(2)
pop_cph_arr = pop_cph.to_numpy()
pop_aar_arr = pop_aar.to_numpy()

diffs = []
for i in range(reps):
    x_cph = rng.choice(pop_cph_arr, size=80, replace=False)
    x_aar = rng.choice(pop_aar_arr, size=120, replace=False)
    diffs.append(x_cph.mean() - x_aar.mean())

formula = np.sqrt(pop_cph_arr.var() / 80 + pop_aar_arr.var() / 120)
print(f"Standard deviation of the 1,000 differences: {np.std(diffs, ddof=1):.3f} hours")
print(f"Formula with the population standard deviations: {formula:.3f} hours")

plt.hist(diffs, bins=20, edgecolor="white")
plt.xlabel("Copenhagen average minus Aarhus average (hours)")
plt.title("1,000 samples of 80 and 120 orders")
plt.show()

### Exercise 2. The survey, analyzed as two groups

**Predict.** The cell treats the 300 before scores and the 300 after scores from Part 1 as two independent groups.
Will the test still reject H<sub>0</sub>? Write down your guess.

**Run** the cell. It prints the standard error, t and p.

In [ ]:
paired = False      # change to True in step 3

after = survey["satisfaction_after"]
before = survey["satisfaction_before"]
n_s = len(survey)
if paired:
    d = after - before
    se_e2 = d.std(ddof=1) / np.sqrt(n_s)
    t_e2 = d.mean() / se_e2
    df_e2 = n_s - 1
else:
    # two groups of equal size: the pooled and the Welch standard error are the same here
    se_e2 = np.sqrt(after.var(ddof=1) / n_s + before.var(ddof=1) / n_s)
    t_e2 = (after.mean() - before.mean()) / se_e2
    df_e2 = 2 * n_s - 2
crit_e2 = stats.t.ppf(0.95, df=df_e2)
print(f"paired = {paired}: se = {se_e2:.4f}, t = {t_e2:.3f}, crit = {crit_e2:.3f}, "
      f"p = {stats.t.sf(t_e2, df=df_e2):.7f}")

**Change** `paired = False` to `True` and run the cell again.

**Answer.** Why is the standard error so much larger without the pairs? Which analysis is right here, and why?

## Part 3. Two proportions

**Question.** RetailCorp tested two versions of its newsletter, assigning recipients to A or B at random. Do the
versions lead to different shares of buyers?

**What we know.** 5,000 recipients per version. The recipients are different people, so the samples are independent.

**What we want.** Whether the sample gives strong evidence that P<sub>B</sub> − P<sub>A</sub> differs from 0.
Two-sided: the marketing team wants to know whether B differs from A in either direction.

**One share for both versions.** In Lecture 3, H<sub>0</sub> gave us the share (P<sub>0</sub> = 0.05). Today
H<sub>0</sub> says that the two shares are equal, but not what they are. If H<sub>0</sub> is true, both versions have
the same share, so we estimate it from all 10,000 recipients together: the pooled proportion
p̂<sub>0</sub> = (x<sub>A</sub> + x<sub>B</sub>) / (n<sub>A</sub> + n<sub>B</sub>), read p-hat-nought, where
x<sub>A</sub> and x<sub>B</sub> are the numbers who bought.

**Hypotheses.** H<sub>0</sub>: P<sub>B</sub> − P<sub>A</sub> = 0 &nbsp;&nbsp;&nbsp; H<sub>1</sub>:
P<sub>B</sub> − P<sub>A</sub> ≠ 0 &nbsp;&nbsp;&nbsp; α = 0.05

**Test statistic.** z = (p̂<sub>B</sub> − p̂<sub>A</sub>) / √(p̂<sub>0</sub>(1 − p̂<sub>0</sub>)(1/n<sub>A</sub> +
1/n<sub>B</sub>)). If the shares are equal and the condition holds, z is approximately standard normal. Condition
(Newbold 10.3): n p̂<sub>0</sub>(1 − p̂<sub>0</sub>) > 5 in each group. Reject H<sub>0</sub> if
z < −z<sub>α/2</sub> or z > z<sub>α/2</sub> = 1.960.

In [ ]:
bought = news.groupby("version")["converted"].sum()     # number who bought, per version
n_A, n_B = 5000, 5000
p_A, p_B = bought["A"] / n_A, bought["B"] / n_B
p0 = (bought["A"] + bought["B"]) / (n_A + n_B)   # pooled
print(f"A: {bought['A']} bought ({p_A:.4f}), B: {bought['B']} bought ({p_B:.4f}), pooled share: {p0:.4f}")
print(f"Condition: {n_A * p0 * (1 - p0):.1f}")    # the same for both groups here

se = np.sqrt(p0 * (1 - p0) * (1 / n_A + 1 / n_B))
z = (p_B - p_A) / se
p = 2 * stats.norm.sf(abs(z))                  # two-sided
print(f"se = {se:.6f}, z = {z:.3f}, crit = {stats.norm.ppf(0.975):.3f}, p = {p:.4f}")

**Decision.** z = 2.886 is above 1.960, and p = 0.0039 is below 0.05, so we reject H<sub>0</sub>.

**For the manager.** 3.9% of the recipients of version B bought something within seven days, against 2.8% of those
who got version A. A difference of 1 percentage point is unlikely to be chance (p = 0.004). Because recipients were
assigned to the versions at random, the difference can be attributed to the newsletter version.

A check with statsmodels:

In [ ]:
from statsmodels.stats.proportion import proportions_ztest

print(proportions_ztest([bought["B"], bought["A"]], [n_B, n_A]))
print(proportions_ztest([bought["A"], bought["B"]], [n_A, n_B]))   # the order of the groups sets the sign

### Which standard error, again?

The test uses the pooled share in the standard error, because H<sub>0</sub> says the shares are equal (Newbold 10.3).
An interval for P<sub>B</sub> − P<sub>A</sub> has no H<sub>0</sub> to assume, so it uses each group's own share. Here
the two are almost equal, because the groups are the same size and the shares are close. They differ more when sizes
and shares differ.

Python and Newbold agree here: `proportions_ztest` with two groups uses the pooled share by default, unlike the
one-sample test in Lecture 3. The order of the groups sets the sign of z.

In [ ]:
se_unpooled = np.sqrt(p_A * (1 - p_A) / n_A + p_B * (1 - p_B) / n_B)
print(f"Pooled (test):      {se:.6f}")
print(f"Unpooled (interval): {se_unpooled:.6f}")

### Exercise 3. The first 2,000 recipients of each version

**Predict.** Suppose the test had stopped after 2,000 recipients per version, with shares of 2.9% and 3.7%. Will it
reject H<sub>0</sub>? Write down your guess.

**Run** the cell.

In [ ]:
n_per_version = 2000      # change to 5000 in step 3

first = news.groupby("version").head(n_per_version)    # the first recipients of each version
b_e3 = first.groupby("version")["converted"].sum()
pA_e3, pB_e3 = b_e3["A"] / n_per_version, b_e3["B"] / n_per_version
p0_e3 = (b_e3["A"] + b_e3["B"]) / (2 * n_per_version)
se_e3 = np.sqrt(p0_e3 * (1 - p0_e3) * (2 / n_per_version))
z_e3 = (pB_e3 - pA_e3) / se_e3
print(f"A: {b_e3['A']} of {n_per_version} ({pA_e3:.4f}), B: {b_e3['B']} of {n_per_version} ({pB_e3:.4f})")
print(f"se = {se_e3:.5f}, z = {z_e3:.3f}, p (two-sided) = {2 * stats.norm.sf(abs(z_e3)):.4f}")

**Change** `n_per_version = 2000` to `5000` and run the cell again.

**Answer.** The shares are similar in both runs. Why does only the full test reject? What does "do not reject" mean
for the smaller test?

## Part 4. Two variances

**Question.** A larger spread means more orders far from the average, so for the same average more late orders. Are
delivery times from Copenhagen more spread out than from Aarhus?

**What we know.** Copenhagen: s = 8.433 hours from 80 orders. Aarhus: s = 7.835 hours from 120 orders. Different
orders in each group, so independent samples. Delivery times are close to normal (Lectures 2 and 3, and the check
below).

**What we want.** Whether the sample gives strong evidence that σ<sub>Cph</sub>² / σ<sub>Aar</sub>², the ratio of the
two population variances, is above 1. For variances we compare a ratio: is one variance a multiple of the other?

**The F distribution.** The test statistic is F = s<sub>1</sub>² / s<sub>2</sub>². If the two population variances
are equal, both populations are normal and the samples are independent, F follows an F distribution with ν<sub>1</sub> = n<sub>1</sub> − 1 and
ν<sub>2</sub> = n<sub>2</sub> − 1 degrees of freedom (ν is read nu, a different symbol from Welch's v in Part 2; numerator first). F is never negative, and the
distribution is skewed to the right. A value near 1 means similar variances.

**Hypotheses.** H<sub>0</sub>: σ<sub>Cph</sub>² ≤ σ<sub>Aar</sub>² &nbsp;&nbsp;&nbsp; H<sub>1</sub>:
σ<sub>Cph</sub>² > σ<sub>Aar</sub>² &nbsp;&nbsp;&nbsp; α = 0.05. Reject H<sub>0</sub> if
F > F<sub>ν1, ν2, α</sub>, with Copenhagen on top.

**The test holds only when both populations are normal.** A large n does not help, and the test is very sensitive to
skewed data (Newbold 10.4). The cell checks the two groups of delivery times.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3), sharex=True)
for ax, x, name in [(axes[0], cph, "Copenhagen"), (axes[1], aar, "Aarhus")]:
    ax.hist(x, bins=15, edgecolor="white")
    ax.set_title(f"{name}: skewness {stats.skew(x):.2f}")
    ax.set_xlabel("Delivery time (hours)")
plt.show()

In [ ]:
s2_cph = cph.var(ddof=1)          # pandas: divides by n - 1
s2_aar = aar.var(ddof=1)
F = s2_cph / s2_aar
nu1, nu2 = len(cph) - 1, len(aar) - 1     # numerator first
F_crit = stats.f.ppf(0.95, nu1, nu2)
p = stats.f.sf(F, nu1, nu2)               # area above F

print(f"s2_cph = {s2_cph:.3f}, s2_aar = {s2_aar:.3f}")
print(f"F = {F:.3f}, crit = {F_crit:.3f}, p = {p:.3f}")

**Decision.** F = 1.158 is below the critical value 1.395, and p = 0.232 is above 0.05, so we do not reject
H<sub>0</sub>.

**For the manager.** Copenhagen delivery times varied a little more in the sample (standard deviation 8.4 against 7.8
hours), but a difference of this size could well be chance (p = 0.23). The sample does not give strong evidence that
Copenhagen is less predictable. It does not show that the two are equally predictable either.

**What the population shows.** In the population, σ = 8.47 hours for Copenhagen and 7.05 hours for Aarhus, a
variance ratio of 1.44. So H<sub>0</sub> is false, and our decision is a Type II error. The reference section has a
simulation of how often the F test detects this difference with 80 and 120 orders.

The F test is a question in its own right. Do not use it to choose between the pooled and the Welch t test: a result
like ours does not show equal variances, and the F test fails with non-normal data. When in doubt, use Welch.

### The F test in Python and in Newbold

* scipy has no function for Newbold's F test of two variances. Compute F and use the F distribution, as above.
* Searching online often leads to `stats.levene` or `stats.bartlett`. These are different tests of equal variances,
with other statistics. They do not give Newbold's F.
* For a two-sided test, Newbold puts the larger variance on top, compares with F at α/2, and doubles the area above F.
With the smaller variance on top, the lower tail gives the same p (Exercise 4).
* Pandas `.var()` divides by n − 1. NumPy's `np.var()` divides by n unless you set `ddof=1`. F uses the sample
variances, with n − 1.

In [ ]:
print(f"{cph.var():.3f} {np.var(cph):.3f} {np.var(cph, ddof=1):.3f}")

### Exercise 4. The two-sided F test

**Predict.** The cell runs the two-sided F test with Copenhagen's larger variance on top. If Aarhus goes on top
instead, what happens to F and to the p-value? Write down your guess.

**Run** the cell.

In [ ]:
top = "Copenhagen"      # change to "Aarhus" in step 3

x1 = cph if top == "Copenhagen" else aar
x2 = aar if top == "Copenhagen" else cph
nu1_e4, nu2_e4 = len(x1) - 1, len(x2) - 1
F_e4 = x1.var(ddof=1) / x2.var(ddof=1)
lower = stats.f.ppf(0.025, nu1_e4, nu2_e4)
upper = stats.f.ppf(0.975, nu1_e4, nu2_e4)
p_e4 = 2 * min(stats.f.sf(F_e4, nu1_e4, nu2_e4), stats.f.cdf(F_e4, nu1_e4, nu2_e4))   # double the smaller tail
print(f"{top} on top: F = {F_e4:.3f}")
print(f"Critical values: {lower:.3f} and {upper:.3f}, p (two-sided) = {p_e4:.3f}")

**Change** `top = "Copenhagen"` to `"Aarhus"` and run the cell again. Was your prediction right?

**Answer.** Why does F change while the p-value stays the same? Why does Newbold put the larger variance on top?

## Reference

Every calculation from the slides is in the parts above. This section adds the F-test simulation and the tables to
revise from.

### How often would the F test detect the difference?

The cell draws 2,000 samples of 80 Copenhagen and 120 Aarhus orders from the population and runs the one-sided F test
at α = 0.05 on each. In the population the variance ratio is 1.44, so every rejection is a correct decision.

In [ ]:
reps = 2000
rng = np.random.default_rng(3)
crit = stats.f.ppf(0.95, 79, 119)
rejections = 0
for i in range(reps):
    x_cph = rng.choice(pop_cph_arr, size=80, replace=False)
    x_aar = rng.choice(pop_aar_arr, size=120, replace=False)
    if np.var(x_cph, ddof=1) / np.var(x_aar, ddof=1) > crit:
        rejections += 1
print(f"The F test rejects H0 in {rejections / reps:.1%} of samples")

### Four tests

Is the gap between two groups, or between before and after, larger than chance would produce?

| Parameter | Samples | Test statistic | Distribution if H<sub>0</sub> is true | Holds when | RetailCorp, α = 0.05 |
|---|---|---|---|---|---|
| μ<sub>d</sub> (Part 1) | Paired: the same units twice | t = d̄ / (s<sub>d</sub>/√n) | t with n − 1 df | Changes normal, or n large | t = 4.892, p < 0.001: reject |
| μ<sub>1</sub> − μ<sub>2</sub> (Part 2) | Independent | t = (x̄<sub>1</sub> − x̄<sub>2</sub>) / √(s<sub>1</sub>²/n<sub>1</sub> + s<sub>2</sub>²/n<sub>2</sub>) | t with v df (Welch) | Both populations normal, or both n large | t = 1.975, p = 0.025: reject |
| P<sub>1</sub> − P<sub>2</sub> (Part 3) | Independent | z = (p̂<sub>1</sub> − p̂<sub>2</sub>) / √(p̂<sub>0</sub>(1 − p̂<sub>0</sub>)(1/n<sub>1</sub> + 1/n<sub>2</sub>)) | Approximately standard normal | n p̂<sub>0</sub>(1 − p̂<sub>0</sub>) > 5 in each group | z = 2.886, p = 0.0039: reject |
| σ<sub>1</sub>² / σ<sub>2</sub>² (Part 4) | Independent | F = s<sub>1</sub>² / s<sub>2</sub>² | F with n<sub>1</sub> − 1 and n<sub>2</sub> − 1 df | Both populations normal. A large n does not help. | F = 1.158, p = 0.232: do not reject |

### Python functions and their defaults

| Test | Function | Watch out for | Set |
|---|---|---|---|
| Paired t | `stats.ttest_rel(after, before)` | Computes first minus second. Two-sided p. | `alternative="greater"` or `"less"`, matching H<sub>1</sub> for that difference |
| Two means | `stats.ttest_ind(x1, x2)` | Pooled by default (`equal_var=True`). Two-sided p. | `equal_var=False` for Welch; `alternative` |
| Two proportions | `proportions_ztest([x1, x2], [n1, n2])` (statsmodels) | Pooled share, as in Newbold. The order sets the sign. Two-sided p. | `alternative="larger"` or `"smaller"` |
| Two variances | None in scipy: `F = s1**2 / s2**2`, then `stats.f.sf(F, nu1, nu2)` | `levene` and `bartlett` are other tests. `np.var` divides by n. | `.var(ddof=1)`; larger variance on top and double the area for two-sided |

Critical values and p-values: `stats.t.ppf(1 - alpha, df)`, `stats.norm.ppf(1 - alpha / 2)` and
`stats.f.ppf(1 - alpha, nu1, nu2)` give upper-tail values; `sf` gives the area above a value and `cdf` the area below
it. For any function, and for code an LLM writes: check what it computes before you use the result.

### Reporting a comparison

Report the size of the difference next to the p-value, and say what the way the data were collected allows you to
conclude.

| Question | Result | What the design allows |
|---|---|---|
| Did satisfaction rise after the redesign? | Up 0.25 points, p < 0.001 | The same customers before and after, but no comparison group: satisfaction rose, and other changes in the period could have contributed |
| Do Copenhagen orders take longer? | 2.3 hours longer, p = 0.025 | Orders were not assigned to warehouses at random: the warehouses differ, and the test does not say why |
| Do the newsletter versions differ? | B 1.0 percentage point higher, p = 0.004 | Random assignment: the difference can be attributed to the version |
| Is Copenhagen less predictable? | Not shown, p = 0.23 | Two separate groups of orders. With 80 and 120 orders a difference of this size is often missed, so the result does not show equal spread. |

### Symbols

| Symbol | Read as | Meaning |
|---|---|---|
| d<sub>i</sub>, d̄, s<sub>d</sub> | d-i, d-bar, s-d | Change for pair i, average change, standard deviation of the changes |
| μ<sub>d</sub> | mu-d | Average change in the population |
| x̄<sub>1</sub>, x̄<sub>2</sub>, s<sub>1</sub>², s<sub>2</sub>² | x-bar one, ..., s two squared | Sample averages and variances of the two groups |
| s<sub>p</sub>² | s-p squared | Pooled variance |
| v | v | Degrees of freedom of Welch's test (Newbold 10.11) |
| p̂<sub>0</sub> | p-hat-nought | Pooled share, if H<sub>0</sub> is true |
| F, ν<sub>1</sub>, ν<sub>2</sub> | F, nu one, nu two | Ratio of two sample variances and its degrees of freedom |

### Danish terms

| English | Danish |
|---|---|
| dependent samples, paired observations | afhængige stikprøver, parrede observationer |
| independent samples | uafhængige stikprøver |
| difference | differens |
| paired t test | parret t-test |
| pooled variance | poolet varians |
| pooled proportion | poolet andel |
| F distribution | F-fordeling |
| numerator, denominator | tæller, nævner |
| variance ratio | variansforhold |